# A FAIR^2 Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets with their @id
record_sets = dataset.metadata.record_sets

print("Available Record Sets:")
for rs in record_sets:
    print(f"- @id: {rs['@id']} | Name: {rs['name']} | Description: {rs['description'] if 'description' in rs else 'No description'}")

# For each record set, list its fields (@id only for consistency)
for rs in record_sets:
    print(f"\nRecord Set: {rs['name']} (@id: {rs['@id']})")
    fields = rs.get('field', [])
    for f in fields:
        print(f"    Field @id: {f['@id']} | Name: {f['name']} | Data Type: {f.get('dataType', 'Unknown')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Example: load all record sets into DataFrames
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Display columns for the first record set
main_rs_id = record_set_ids[0]
print(f"Columns in record set {main_rs_id}: {dataframes[main_rs_id].columns.tolist()}")
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Let's pick a numeric field and a group/categorical field from the main record set
# We'll use the @id values for consistency

# Example field IDs (replace with actual field @ids from overview if their content is available)
# For this workbook, we'll assume the primary record set and field names (for e.g. GAD7_score, village) are:
#  - numeric_field_id = '@id:GAD7_score'
#  - group_field_id = '@id:village'
# If different, copy those @ids from the previous Data Overview cell.

df = dataframes[main_rs_id]

# Guess candidate fields
candidate_numeric_fields = [col for col in df.columns if 'gad' in col.lower() or 'phq' in col.lower() or 'psq' in col.lower() or 'score' in col.lower()]
candidate_group_fields = [col for col in df.columns if 'village' in col.lower() or 'education' in col.lower() or 'marital' in col.lower()]

print(f"Numeric candidates: {candidate_numeric_fields}")
print(f"Grouping candidates: {candidate_group_fields}")

# Choose the first matching field
numeric_field = candidate_numeric_fields[0] if candidate_numeric_fields else df.columns[0]
group_field = candidate_group_fields[0] if candidate_group_fields else df.columns[0]

threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Grouping
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of normalized numeric field
plt.figure(figsize=(8,4))
sns.histplot(filtered_df[f"{numeric_field}_normalized"], bins=15, kde=True)
plt.title(f'Normalized Distribution of {numeric_field}')
plt.xlabel(f'{numeric_field}_normalized')
plt.ylabel('Count')
plt.show()

# Group-wise mean score barplot
if group_field in filtered_df.columns:
    plt.figure(figsize=(10,4))
    sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
    plt.title(f'Mean {numeric_field} by {group_field}')
    plt.xticks(rotation=45)
    plt.xlabel(group_field)
    plt.ylabel(f'Mean {numeric_field}')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 Mental Health Survey Dataset provides diverse mental health and demographic indicators for Kilifi County, Kenya.
- Using `mlcroissant`, we loaded and explored multiple record sets by their `@id` and demonstrated filtering, normalization, and grouping operations, referencing entity fields via their unique `@id` throughout.
- Visualizations revealed distribution of scores and differences across demographic groups. Further analyses could include correlations between survey metrics, deeper missing data profiling, and stratification by additional attributes.